# DeMemte Transformer Only

Notebook reducido para entrenar y evaluar exclusivamente la variante `dememte_transformer` basada en:
- `AttentionSpatialVQVAE`
- `DeMemteSpatial`

In [ ]:
import os
import copy
import json
import random
from dataclasses import dataclass, asdict
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset, Subset

import torchvision
import torchvision.transforms as transforms
from sklearn.model_selection import StratifiedShuffleSplit

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('cuda:', torch.cuda.is_available())

In [ ]:
@dataclass
class Config:
    data_dir: str = '../data'
    num_classes: int = 102
    batch_size: int = 16
    num_workers: int = 2

    val_ratio: float = 0.2
    split_seed: int = 42

    lr_vq: float = 3e-4
    lr_cls: float = 1e-4
    weight_decay: float = 1e-4

    epochs_phase1_max: int = 4
    epochs_phase2_max: int = 10

    early_stop_patience: int = 3
    early_stop_min_delta: float = 1e-4

    scheduler_factor: float = 0.5
    scheduler_patience: int = 1

    embedding_dim: int = 256
    num_embeddings: int = 1024
    commitment_cost: float = 0.25

    init_tau: float = 1.0
    init_alpha: float = 1.5

    denoise_weight: float = 0.5
    vq_weight: float = 0.25

    attn_heads: int = 4
    attn_layers: int = 2
    masked_feature_ratio: float = 0.35

    train_corrupt_prob: float = 0.7

    max_train_batches_debug: int = 10000000
    max_val_batches_debug: int = 10000000

    artifacts_root: str = './out/artifacts'
    experiment_name: str = 'dememte_transformer_only'

    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = Config()
device = torch.device(cfg.device)
criterion = nn.CrossEntropyLoss()
print(json.dumps(asdict(cfg), indent=2))

In [ ]:
def _extract_labels(dataset):
    if hasattr(dataset, '_labels'):
        return np.array(dataset._labels)
    if hasattr(dataset, 'labels'):
        return np.array(dataset.labels)
    ys = []
    for i in range(len(dataset)):
        _, y = dataset[i]
        ys.append(int(y))
    return np.array(ys)

def build_datasets(config: Config):
    transform_train = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    transform_eval = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    tr = torchvision.datasets.Flowers102(root=config.data_dir, split='train', download=False, transform=transform_train)
    va = torchvision.datasets.Flowers102(root=config.data_dir, split='val', download=False, transform=transform_train)
    te = torchvision.datasets.Flowers102(root=config.data_dir, split='test', download=False, transform=transform_eval)

    cv_ds = ConcatDataset([tr, va])
    cv_y = np.concatenate([_extract_labels(tr), _extract_labels(va)], axis=0)

    splitter = StratifiedShuffleSplit(n_splits=1, test_size=config.val_ratio, random_state=config.split_seed)
    train_idx, val_idx = next(splitter.split(np.zeros(len(cv_y)), cv_y))

    tr_ds = Subset(cv_ds, train_idx.tolist())
    va_ds = Subset(cv_ds, val_idx.tolist())

    tr_loader = DataLoader(tr_ds, batch_size=config.batch_size, shuffle=True, num_workers=config.num_workers, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=config.batch_size, shuffle=False, num_workers=config.num_workers, pin_memory=True)
    te_loader = DataLoader(te, batch_size=config.batch_size, shuffle=False, num_workers=config.num_workers, pin_memory=True)

    return tr_loader, va_loader, te_loader

trainloader, valloader, testloader = build_datasets(cfg)
print('train batches:', len(trainloader), '| val batches:', len(valloader), '| test batches:', len(testloader))

In [ ]:
def make_backbone():
    base = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
    return nn.Sequential(*list(base.children())[:-2])

class VectorQuantizer2D(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.embedding.weight.data.uniform_(-1 / num_embeddings, 1 / num_embeddings)

    def forward(self, z_e):
        b, c, h, w = z_e.shape
        z_e_perm = z_e.permute(0, 2, 3, 1).contiguous()
        flat = z_e_perm.view(-1, c)
        emb = self.embedding.weight
        distances = flat.pow(2).sum(1, keepdim=True) - 2 * flat @ emb.t() + emb.pow(2).sum(1, keepdim=True).t()
        idx = torch.argmin(distances, dim=1)
        one_hot = F.one_hot(idx, self.num_embeddings).type(flat.dtype)
        q_flat = one_hot @ emb
        q = q_flat.view(b, h, w, c).permute(0, 3, 1, 2).contiguous()
        vq_loss = self.commitment_cost * F.mse_loss(z_e, q.detach()) + F.mse_loss(q, z_e.detach())
        q_st = z_e + (q - z_e).detach()
        dq_map = ((z_e - q.detach()) ** 2).mean(dim=1, keepdim=True)
        return q_st, vq_loss, dq_map

class AttentionSpatialVQVAE(nn.Module):
    def __init__(self, in_channels=512, hidden_channels=256, embedding_dim=128, num_embeddings=512, commitment_cost=0.25, heads=4, layers=1):
        super().__init__()
        self.pre = nn.Sequential(
            nn.Conv2d(in_channels, hidden_channels, 1),
            nn.BatchNorm2d(hidden_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, embedding_dim, 1),
        )
        self.pos_embed = nn.Parameter(torch.zeros(1, embedding_dim, 7, 7))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.attn_blocks = nn.ModuleList([
            nn.MultiheadAttention(embed_dim=embedding_dim, num_heads=heads, batch_first=True) for _ in range(layers)
        ])
        self.attn_norms = nn.ModuleList([nn.LayerNorm(embedding_dim) for _ in range(layers)])
        self.ffn_blocks = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embedding_dim, embedding_dim * 4),
                nn.GELU(),
                nn.Linear(embedding_dim * 4, embedding_dim),
            ) for _ in range(layers)
        ])
        self.ffn_norms = nn.ModuleList([nn.LayerNorm(embedding_dim) for _ in range(layers)])

        self.vq = VectorQuantizer2D(num_embeddings, embedding_dim, commitment_cost)
        self.post = nn.Sequential(
            nn.Conv2d(embedding_dim, hidden_channels, 1),
            nn.BatchNorm2d(hidden_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, in_channels, 1),
        )

    def _run_attn(self, z):
        b, c, h, w = z.shape
        pos = self.pos_embed
        if pos.shape[-2:] != (h, w):
            pos = F.interpolate(pos, size=(h, w), mode='bilinear', align_corners=False)
        z = z + pos

        toks = z.flatten(2).transpose(1, 2)
        for attn, attn_norm, ffn, ffn_norm in zip(self.attn_blocks, self.attn_norms, self.ffn_blocks, self.ffn_norms):
            out, _ = attn(toks, toks, toks, need_weights=False)
            toks = attn_norm(toks + out)
            ff = ffn(toks)
            toks = ffn_norm(toks + ff)

        z2 = toks.transpose(1, 2).view(b, c, h, w)
        return z2

    def forward(self, x):
        z = self.pre(x)
        z = self._run_attn(z)
        zq, vq_loss, dq_map = self.vq(z)
        x_rec = self.post(zq)
        return x_rec, vq_loss, dq_map

class DeMemteSpatial(nn.Module):
    def __init__(self, backbone, vq_vae, num_classes):
        super().__init__()
        self.backbone = backbone
        self.vq_vae = vq_vae
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(512, num_classes)
        self.gate_tau = nn.Parameter(torch.tensor(0.0))
        self.gate_alpha = nn.Parameter(torch.tensor(1.5))
        self.register_buffer('dq_ema_mean', torch.tensor(0.0))
        self.register_buffer('dq_ema_var', torch.tensor(1.0))
        self.register_buffer('dq_ema_counted', torch.tensor(0.0))

    def set_backbone_trainable(self, trainable: bool):
        for p in self.backbone.parameters():
            p.requires_grad = trainable

    def _update_ema(self, dq_map):
        with torch.no_grad():
            batch_mean = dq_map.mean()
            batch_var = dq_map.var(unbiased=False) if dq_map.numel() > 1 else torch.tensor(0.0, device=dq_map.device)
            momentum = 0.99
            if self.dq_ema_counted.item() == 0:
                self.dq_ema_mean.fill_(batch_mean)
                self.dq_ema_var.fill_(batch_var)
            else:
                self.dq_ema_mean.mul_(momentum).add_(batch_mean * (1 - momentum))
                self.dq_ema_var.mul_(momentum).add_(batch_var * (1 - momentum))
            self.dq_ema_counted.add_(1.0)

    def _mask_features(self, feats, mask_ratio):
        if mask_ratio <= 0:
            return feats
        b, _, h, w = feats.shape
        keep = (torch.rand(b, 1, h, w, device=feats.device) > mask_ratio).float()
        return feats * keep

    def forward(self, x, target_feats=None, update_ema=True, return_debug=False, feature_mask_ratio=0.0):
        input_feats = self.backbone(x)
        vq_input = self._mask_features(input_feats, feature_mask_ratio if self.training else 0.0)
        rec_feats, vq_loss, dq_map = self.vq_vae(vq_input)

        target = input_feats.detach() if target_feats is None else target_feats.detach()
        denoise_loss = F.mse_loss(rec_feats, target)

        if self.training and update_ema:
            self._update_ema(dq_map)

        dq_norm = (dq_map - self.dq_ema_mean) / (torch.sqrt(self.dq_ema_var) + 1e-5)
        alpha = F.softplus(self.gate_alpha)
        signal = torch.sigmoid(alpha * (dq_norm - self.gate_tau))
        enhanced = (1.0 - signal) * input_feats + signal * rec_feats

        logits = self.classifier(self.pool(enhanced).flatten(1))
        if return_debug:
            return logits, denoise_loss, vq_loss, {'clean_feats': input_feats, 'enhanced_feats': enhanced, 'signal': signal}
        return logits, denoise_loss, vq_loss

In [ ]:
def make_transformer_variant(config: Config):
    backbone = make_backbone()
    model = DeMemteSpatial(
        backbone,
        AttentionSpatialVQVAE(
            in_channels=512,
            hidden_channels=256,
            embedding_dim=config.embedding_dim,
            num_embeddings=config.num_embeddings,
            commitment_cost=config.commitment_cost,
            heads=config.attn_heads,
            layers=config.attn_layers,
        ),
        config.num_classes,
    ).to(device)

    model.gate_tau.data.fill_(config.init_tau)
    model.gate_alpha.data.fill_(config.init_alpha)

    vq_params = list(model.vq_vae.parameters())
    cls_params = [model.gate_tau, model.gate_alpha, model.classifier.weight, model.classifier.bias]

    model.set_backbone_trainable(False)

    optimizer_p1 = optim.AdamW(vq_params, lr=config.lr_vq, weight_decay=config.weight_decay)
    scheduler_p1 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_p1, mode='min', factor=config.scheduler_factor, patience=config.scheduler_patience)

    optimizer_p2 = optim.AdamW([
        {'params': vq_params, 'lr': config.lr_vq},
        {'params': cls_params, 'lr': config.lr_cls},
    ], weight_decay=config.weight_decay)
    scheduler_p2 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_p2, mode='max', factor=config.scheduler_factor, patience=config.scheduler_patience)

    return model, optimizer_p1, scheduler_p1, optimizer_p2, scheduler_p2

def apply_train_corruption(x):
    if random.random() > cfg.train_corrupt_prob:
        return x

    mode = random.choice(['gaussian_noise', 'pixel_mask', 'cutout', 'blur'])

    if mode == 'gaussian_noise':
        level = random.uniform(0.4, 1.3)
        return x + level * torch.randn_like(x)

    if mode == 'pixel_mask':
        level = random.uniform(0.20, 0.65)
        keep = (torch.rand(x.size(0), 1, x.size(2), x.size(3), device=x.device) > level).float()
        return x * keep

    if mode == 'cutout':
        level = random.uniform(0.20, 0.45)
        b, _, h, w = x.shape
        cut_h = max(1, int(h * level))
        cut_w = max(1, int(w * level))
        mask = torch.ones((b, 1, h, w), device=x.device, dtype=x.dtype)
        for i in range(b):
            t = random.randint(0, max(0, h - cut_h))
            l = random.randint(0, max(0, w - cut_w))
            mask[i, :, t:t + cut_h, l:l + cut_w] = 0.0
        return x * mask

    level = random.uniform(0.30, 0.80)
    channels = x.size(1)
    kernel_size = 7
    weight = torch.ones(channels, 1, kernel_size, kernel_size, device=x.device, dtype=x.dtype) / (kernel_size * kernel_size)
    blur = F.conv2d(x, weight, padding=kernel_size // 2, groups=channels)
    return (1.0 - level) * x + level * blur

def run_epoch_phase1(model, loader, optimizer, train):
    model.train(train)
    model.set_backbone_trainable(False)
    totals = {'loss': 0.0, 'n': 0}

    for bi, (x, _) in enumerate(loader):
        if train and bi >= cfg.max_train_batches_debug:
            break
        if (not train) and bi >= cfg.max_val_batches_debug:
            break

        x = x.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        _, denoise_loss, vq_loss = model(x, feature_mask_ratio=cfg.masked_feature_ratio)
        loss = cfg.denoise_weight * denoise_loss + cfg.vq_weight * vq_loss

        if train:
            loss.backward()
            optimizer.step()

        bs = x.size(0)
        totals['loss'] += loss.item() * bs
        totals['n'] += bs

    totals['loss'] /= max(1, totals['n'])
    return totals

def run_epoch_phase2(model, loader, optimizer, train):
    model.train(train)
    model.set_backbone_trainable(False)
    totals = {'loss': 0.0, 'acc': 0.0, 'n': 0}

    for bi, (x, y) in enumerate(loader):
        if train and bi >= cfg.max_train_batches_debug:
            break
        if (not train) and bi >= cfg.max_val_batches_debug:
            break

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        x_clean = x
        x_noisy = apply_train_corruption(x.clone()) if train else x

        if train:
            optimizer.zero_grad(set_to_none=True)

        logits_clean, denoise_clean, vq_clean, dbg_clean = model(x_clean, target_feats=None, update_ema=True, return_debug=True, feature_mask_ratio=cfg.masked_feature_ratio)
        clean_target_feats = dbg_clean['clean_feats'].detach()

        logits_noisy, denoise_noisy, vq_noisy = model(x_noisy, target_feats=clean_target_feats, update_ema=False, feature_mask_ratio=cfg.masked_feature_ratio)
        ce_noisy = criterion(logits_noisy, y)

        denoise_loss = 0.5 * (denoise_clean + denoise_noisy)
        vq_loss = 0.5 * (vq_clean + vq_noisy)
        loss = ce_noisy + cfg.denoise_weight * denoise_loss + cfg.vq_weight * vq_loss

        if train:
            loss.backward()
            optimizer.step()

        bs = x.size(0)
        acc_clean = (logits_clean.argmax(1) == y).float().mean().item()
        acc_noisy = (logits_noisy.argmax(1) == y).float().mean().item()
        totals['loss'] += loss.item() * bs
        totals['acc'] += (0.5 * (acc_clean + acc_noisy)) * bs
        totals['n'] += bs

    totals['loss'] /= max(1, totals['n'])
    totals['acc'] /= max(1, totals['n'])
    return totals

@torch.no_grad()
def eval_clean_acc(model, loader, max_batches=10000000):
    model.eval()
    correct, total = 0, 0
    for bi, (x, y) in enumerate(loader):
        if bi >= max_batches:
            break
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits, _, _ = model(x)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / max(1, total)

In [ ]:
run_stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
artifacts_dir = os.path.join(cfg.artifacts_root, f'{cfg.experiment_name}_{run_stamp}')
os.makedirs(artifacts_dir, exist_ok=True)

print('===== Train dememte_transformer =====')
model, opt1, sch1, opt2, sch2 = make_transformer_variant(cfg)

best_p1, best_p1_state, no_imp1 = float('inf'), copy.deepcopy(model.state_dict()), 0
for ep in range(1, cfg.epochs_phase1_max + 1):
    tr = run_epoch_phase1(model, trainloader, opt1, train=True)
    va = run_epoch_phase1(model, valloader, opt1, train=False)
    sch1.step(va['loss'])
    print(f'[dememte_transformer P1 {ep:02d}] tr_loss={tr["loss"]:.4f} val_loss={va["loss"]:.4f}')

    if va['loss'] < best_p1 - cfg.early_stop_min_delta:
        best_p1 = va['loss']
        best_p1_state = copy.deepcopy(model.state_dict())
        no_imp1 = 0
    else:
        no_imp1 += 1
        if no_imp1 >= cfg.early_stop_patience:
            print(f'P1 early stop en epoca {ep}.')
            break

model.load_state_dict(best_p1_state)

best_p2, best_p2_state, no_imp2 = -1.0, copy.deepcopy(model.state_dict()), 0
for ep in range(1, cfg.epochs_phase2_max + 1):
    tr = run_epoch_phase2(model, trainloader, opt2, train=True)
    va = run_epoch_phase2(model, valloader, opt2, train=False)
    sch2.step(va['acc'])
    print(f'[dememte_transformer P2 {ep:02d}] tr_acc={tr["acc"]:.4f} val_acc={va["acc"]:.4f}')

    if va['acc'] > best_p2 + cfg.early_stop_min_delta:
        best_p2 = va['acc']
        best_p2_state = copy.deepcopy(model.state_dict())
        no_imp2 = 0
    else:
        no_imp2 += 1
        if no_imp2 >= cfg.early_stop_patience:
            print(f'P2 early stop en epoca {ep}.')
            break

model.load_state_dict(best_p2_state)
val_acc = eval_clean_acc(model, valloader)
test_clean_acc = eval_clean_acc(model, testloader)

results = {
    'variant': 'dememte_transformer',
    'val_acc': float(val_acc),
    'test_clean_acc': float(test_clean_acc),
}

summary = {
    'config': asdict(cfg),
    'results': results,
}

summary_path = os.path.join(artifacts_dir, 'dememte_transformer_summary.json')
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

torch.save({
    'name': 'dememte_transformer',
    'state_dict': model.state_dict(),
    'config': asdict(cfg),
}, os.path.join(artifacts_dir, 'dememte_transformer_best.pt'))

print('Resultados:', results)
print('artifact dir:', artifacts_dir)
print('summary:', summary_path)